# 05 · NGSIM stop-and-go — Acceleration

Two-shot decomposition of follower **acceleration** for 16 stop-and-go
car-following trajectories extracted from NGSIM (`data/ngsim/stop_to_go.csv`).
Trajectories are resampled to the length of the shortest one (context alignment),
decomposed into common ($C\Psi$) and personalized ($P\Phi$) components, and the
personalized components are clustered with k-means (k = 3).

**Figures produced:** `NGSIM_accel_component.png`, `NGSIM_accel_cluster.png`,
`NGSIM_cluster_accel.png`.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "bhd").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

from bhd import DATA_DIR, FIG_DIR, set_style, load_ngsim_stop_to_go

set_style(seaborn_white=True)

## 1. Load and align trajectories

In [ ]:
df = load_ngsim_stop_to_go(DATA_DIR / "ngsim" / "stop_to_go.csv")
traj_ids = sorted(df["trajectory_id"].unique())
min_len = min(len(df[df["trajectory_id"] == tid]) for tid in traj_ids)
print(f"{len(traj_ids)} trajectories, resampled to {min_len} points (0.1 s per step)")

acc_resampled = []
for tid in traj_ids:
    acc = df[df["trajectory_id"] == tid]["follower_acc"].values
    acc_resampled.append(np.interp(np.linspace(0, 1, min_len), np.linspace(0, 1, len(acc)), acc))
A = np.stack(acc_resampled, axis=1)  # (time, vehicles)

## 2. Two-shot decomposition

In [ ]:
pca_shared = PCA(n_components=1)
C = pca_shared.fit(A.T).components_.T
Psi = C.T @ A
A_shared = C @ Psi

personalized, reconstructed = [], []
for v in range(A.shape[1]):
    R_v = (A[:, v] - A_shared[:, v]).reshape(-1, 1)
    pca_personal = PCA(n_components=1)
    Phi_v = pca_personal.fit_transform(R_v)
    R_hat = pca_personal.inverse_transform(Phi_v)
    personalized.append(R_hat.flatten())
    reconstructed.append(A_shared[:, v] + R_hat.flatten())
A_personal = np.column_stack(personalized)
A_reconstructed = np.column_stack(reconstructed)

colors = cm.viridis(np.linspace(0, 1, A.shape[1]))
xticks = list(range(0, 401, 50))
xtick_labels = [str(x // 10) for x in xticks]

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
for i in range(A.shape[1]):
    axes[0].plot(A[:, i], color=colors[i], alpha=0.4)
    axes[1].plot(A_shared[:, i], color=colors[i], alpha=0.6)
    axes[2].plot(A_personal[:, i], linestyle="--", color=colors[i], alpha=0.8)
axes[0].set_title("True Acceleration", fontsize=22)
axes[1].set_title("Common Component (CΨ)", fontsize=22)
axes[2].set_title("Personalized Component (PΦ)", fontsize=22)
axes[2].set_xlabel("Time (sec)", fontsize=22)
for ax in axes:
    ax.tick_params(axis="both", labelsize=20)
    ax.set_ylim(-10, 10)
    ax.set_xlim(0, 400)
    ax.set_yticks([-10, -5, 0, 5, 10])
    ax.set_ylabel("Acceleration (m/s²)", fontsize=20)
    ax.set_xticks(xticks)
    ax.set_xticklabels(xtick_labels)
    ax.grid(True)
plt.tight_layout()
plt.savefig(FIG_DIR / "NGSIM_accel_component.png", dpi=300, bbox_inches="tight")
plt.show()

## 3. Clustering the personalized components

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = kmeans.fit_predict(A_personal.T)
print("Cluster sizes:", np.bincount(labels))

plt.figure(figsize=(10, 7))
palette = sns.color_palette("Set2", n_colors=3)
for cid in range(3):
    idx = np.where(labels == cid)[0]
    plt.plot(A_personal[:, idx].mean(axis=1), label=f"Cluster {cid}", color=palette[cid], linewidth=3)
plt.xlabel("Time (sec)", fontsize=22)
plt.xticks(ticks=xticks, labels=xtick_labels, fontsize=20)
plt.xlim(0, 400)
plt.ylabel("Acceleration (m/s²)", fontsize=22)
plt.yticks(fontsize=20)
plt.legend(fontsize=22)
plt.grid(True)
plt.tight_layout()
plt.savefig(FIG_DIR / "NGSIM_accel_cluster.png", dpi=300, bbox_inches="tight")
plt.show()

pca_2d = PCA(n_components=2)
X_vis = pca_2d.fit_transform(A_personal.T)
plt.figure(figsize=(10, 10))
sns.scatterplot(x=X_vis[:, 0], y=X_vis[:, 1], hue=labels, palette="Set2", s=120, edgecolor="black")
for i in range(len(labels)):
    plt.text(X_vis[i, 0] + 0.02, X_vis[i, 1], f"V{i + 1}", fontsize=20)
handles, legend_labels = plt.gca().get_legend_handles_labels()
new_labels = [f"Cluster {int(lbl)}" if lbl.isdigit() else lbl for lbl in legend_labels]
plt.xlabel("Personalized Component 1", fontsize=22)
plt.xticks(fontsize=20)
plt.ylabel("Personalized Component 2", fontsize=22)
plt.yticks(fontsize=20)
plt.legend(handles, new_labels, fontsize=22)
plt.grid(True)
plt.tight_layout()
plt.savefig(FIG_DIR / "NGSIM_cluster_accel.png", dpi=300, bbox_inches="tight")
plt.show()